# 结构化掩码可视化：Seg / ADE20K（FCN-R50 + EDEN）
**回答的问题：各训练阶段的 ckpt 当前会怎么蒙。**

轻量版（G ~ IP 的金标准裁判见 probe_mask_meta.ipynb，这里不碰去噪器、成本低两个量级）：
G = |∂L_task/∂x₀| -> 划块取均值 -> 逐图 softmax 采样概率 -> Gumbel-top-k 无放回定量
现采掩码 -> 蒙后输入 x̃₀（与训练 MaskSelector / MaskApplier 同口径同公式）。

CRN：跨 ckpt 固定同批 val 图、同组采样噪声（专用种子），严格可比。
GUIDED=True 展示"该阶段 ckpt 的 G 引导选址"（预热期的均匀随机行为用 GUIDED=False 出图）。
每个 ckpt 出一张五联图（原图 | G 热力 | 采样概率 | keep-mask | 蒙后输入），
跨 ckpt 出演化拼图（行=训练阶段，列=同一批图）；测量结果缓存 npz，复画免重测。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把路径算成绝对路径 ----
VISUAL = os.path.abspath('.')
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAE）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import importlib
import probe_core as pc
importlib.reload(pc)          # 防止内核 sys.modules 驻留旧版 probe_core
assert getattr(pc, 'MASK_VIZ_VERSION', 0) >= 3, \
    'probe_core 是旧版（无 MASK_VIZ_VERSION>=2 的掩码可视化）——请确认 visual/probe_core.py 已同步并重启内核'

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config（口径与训练一致）；
# UNet 仪器的旧 run 需把 UNET_FIX 改为 dict(denoiser_type='unet')。
import glob, re
RUN_DIR = 'work_dirs/eden/fcn_diffae_tweight_mask'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]
UNET_FIX = None
CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/mask_viz'
NUM_IMAGES = 8        # 测量/统计用图数（跨 ckpt 固定）
DISPLAY_IMAGES = 4    # 五联图/演化拼图只画前几张
MASK_RATIO = None     # 蒙块量；None = 模型配置值（未启用掩码的旧 run 回退 0.25）
GUIDED = True         # True = G 引导选址；False = 预热期均匀随机（同 k 块、位置随机）
SEED = 0
os.makedirs(OUT, exist_ok=True)
CKPTS

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG, cfg_overrides=UNET_FIX)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
batch['img01'] = pc.img01_from_xis(batch['xis'])           # [0,1] 展示图
del model; torch.cuda.empty_cache()
# GT 标注：分割真值半透明叠加 + 类边界 + 主类图例（ADE20K 官方 palette）
from mmseg.datasets import ADE20KDataset
_meta = ADE20KDataset.METAINFO
ann_fn = lambda im, i: pc.annotate_seg(im, batch['dsis'][i].gt_sem_seg.data,
                                       palette=_meta['palette'],
                                       names=_meta['classes'])

In [ ]:
results, labels = [], []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt, cfg_overrides=UNET_FIX)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    # seg 逐图反传（val 图尺寸不一，不能 cat）：task loss = decode head 训练 loss
    grad_fn = lambda xg, dsi: pc.make_task_loss_fn(model, [xg], [dsi])()
    res = pc.measure_mask_viz(model, batch, grad_fn, batch_grad=False,
                              mask_ratio=MASK_RATIO, guided=GUIDED, seed=SEED,
                              ann_fn=ann_fn)
    pc.save_mask_viz_cache(os.path.join(OUT, f'{tag}_cache.npz'), res)
    fig = pc.plot_mask_viz(res, title=tag, max_images=DISPLAY_IMAGES,
                           save_path=os.path.join(OUT, f'{tag}_mask_viz.png'))
    results.append(res); labels.append(tag)
    print(f'{tag}: masked={res["masked_frac"].mean():.2f} '
          f'(k={res["k"]}/{res["g"]**2}, alpha={res["alpha"]:g})')
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 演化拼图：行 = 训练阶段，列 = 同一批图
for mode in ['masked', 'prob', 'g']:
    pc.plot_mask_viz_evolution(results, labels, mode=mode,
                               max_images=DISPLAY_IMAGES,
                               save_path=os.path.join(OUT, f'evolution_{mode}.png'))
print('saved evolution_{masked,prob,g}.png')

In [ ]:
# 免重测复画：从 npz 缓存重建 res（换 DISPLAY_IMAGES / 标题都行；掩码已在缓存里）
REPLAY_TAG = os.path.splitext(os.path.basename(CKPTS[-1]))[0]  # 改成想复画的 ckpt tag
res = pc.load_mask_viz_cache(os.path.join(OUT, f'{REPLAY_TAG}_cache.npz'))
fig = pc.plot_mask_viz(res, title=f'{REPLAY_TAG} (replay)',
                       max_images=DISPLAY_IMAGES,
                       save_path=os.path.join(OUT, f'{REPLAY_TAG}_replay.png'))
fig